# Error Case 15 — 500/503: PostgreSQL Connection Pool Failure

**Error:** `500 Internal Server Error` / `503 Service Unavailable`  
**Root cause:** PostgreSQL down → all connections broken → pool exhausted  
**Fix:** PostgreSQL recovery + Polaris pod restart  

## Flow
```
PostgreSQL goes down
  → Existing connections: TCP reset / broken pipe
  → New connection attempts: connection refused
  → Agroal pool: marks connections invalid
  → Polaris requests: 500 (SQL error) or 503 (pool exhausted)

PostgreSQL comes back
  → Pool auto-creates new connections
  → BUT: if restored from backup → entity_version mismatch
  → See Notebook 16 for that case
```

In [1]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import sys
import importlib
import polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *
require_not_prod("error-case test")  # PROD-safe: refuses to run against company PROD

cleanup()
print('✅ Setup complete')

# Verify Polaris healthy before test
r = requests.get(f'{BASE_MGMT}/catalogs', headers=h())
print(f'Polaris health before test: {r.status_code}')
assert r.status_code == 200

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🧹 Cleaning up all test resources...
✅ Cleanup done
✅ Setup complete
Polaris health before test: 200


In [4]:
# Cell 1.5: Provision the DB Connection Pool OpenSearch monitor (idempotent)
# Registers Monitor 7 from error-cases/opensearch-monitor-alert-guide.md via
# the Alerting REST API, so the outage below is caught by a REAL monitor +
# trigger, not just proven present in raw logs.
import opensearch_alert_provisioner as alert_prov

try:
    db_pool_monitor_id = alert_prov.provision_db_connection_pool_monitor()
    print(f'✅ Monitor ready: {db_pool_monitor_id}')
except RuntimeError as e:
    db_pool_monitor_id = None
    print(f'⚠️  Skipping monitor provisioning: {e}')
    print('   (Create the "Polaris SMS Webhook" destination manually first — guide Step 1)')

⚠️  Skipping monitor provisioning: Notification destination "Polaris SMS Webhook" not found. Create it manually first (guide Step 1, Alerting -> Notifications -> Create channel) before provisioning the monitor — this module intentionally does not create destinations pointed at the internal Kafka webhook.
   (Create the "Polaris SMS Webhook" destination manually first — guide Step 1)


In [5]:
# Cell 2: Get token FIRST, then simulate outage
import subprocess, time

# ── Step 1: Get token while DB still up ──────────────────────
pre_outage_token = root_token()
print(f'✅ Token obtained before outage')
print(f'   Token valid for 1 hour — will use during outage test')

# ── Step 2: Scale down pgpool ────────────────────────────────
print('\n💥 Simulating PostgreSQL outage...')
result = subprocess.run([
    'kubectl', 'scale', 'deployment',
    'benchmarks-postgresql-postgresql-ha-pgpool',
    '--replicas=0',
    '-n', 'datahub-hynix'
], capture_output=True, text=True)

print(result.stdout or result.stderr)

time.sleep(5)
print('pgpool scaled down — PostgreSQL unreachable')

✅ Token obtained before outage
   Token valid for 1 hour — will use during outage test

💥 Simulating PostgreSQL outage...
deployment.apps/benchmarks-postgresql-postgresql-ha-pgpool scaled

pgpool scaled down — PostgreSQL unreachable


In [7]:
# Verify Polaris is unhealthy before running outage test
import requests, time

print('Verifying Polaris is unhealthy...')

# Try up to 10s for Polaris to detect DB is gone
for attempt in range(5):
    try:
        r = requests.get(
            f'{BASE_MGMT}/catalogs',
            headers={
                'Authorization': f'Bearer {pre_outage_token}',
                'Polaris-Realm': REALM,
            },
            timeout=5
        )
        print(f'Attempt {attempt+1}: {r.status_code}')
        if r.status_code in [500, 503]:
            print('✅ Polaris confirmed unhealthy — DB unreachable')
            break
        elif r.status_code == 200:
            print('⚠️  Polaris still healthy — pool has cached connections')
            print('   Waiting for pool to exhaust...')
            time.sleep(3)
    except Exception as e:
        print(f'Attempt {attempt+1}: {type(e).__name__} — Polaris unreachable ✅')
        break
else:
    print('⚠️  Polaris still responding — proceed anyway')
    print('   Connection pool may still have valid connections')
    print('   Need more requests to exhaust pool')

print(f'pre_outage_token: {pre_outage_token[:20]}...')
assert pre_outage_token and len(pre_outage_token) > 50, 'Token is empty or invalid!'

Verifying Polaris is unhealthy...
Attempt 1: 401
Attempt 2: 401
Attempt 3: ReadTimeout — Polaris unreachable ✅
pre_outage_token: eyJhbGciOiJSUzI1NiIs...


In [8]:
# Cell 2.5: Wait for Agroal connection error log
# Agroal detects DB unreachable on background validation
# This appears WITHOUT any API request — proactive detection

import time

print('Waiting for Agroal to detect connection failure...')
print('Expected log: io.agroal.pool WARN Connection refused')
print()

# Wait for Agroal background thread to detect
time.sleep(15)  # Agroal checks periodically

# Search OpenSearch for Agroal connection error
hits = search_db_errors(minutes_ago=1)

if hits:
    print('✅ Agroal detected DB failure proactively:')
    print_logs(hits)
    print()
    print('Key insight:')
    print('  → This log appears WITHOUT any API request')
    print('  → Agroal background thread validates connections')
    print('  → Detects failure before any user request hits broken connection')
    print('  → OpenSearch alert can fire BEFORE users see errors')
else:
    print('⚠️  Agroal log not found yet — try increasing wait time')
    print('Checking kubectl logs directly...')
    import subprocess
    result = subprocess.run([
        'kubectl', 'logs', '-n', 'datahub-hynix',
        'deployment/benchmarks-polaris',
        '--tail=10'
    ], capture_output=True, text=True)
    print(result.stdout)

Waiting for Agroal to detect connection failure...
Expected log: io.agroal.pool WARN Connection refused

✅ Agroal detected DB failure proactively:
Found 10 log entries:

⚠️  [2026-07-03T04:44:09.934Z] WARN
   logger    : io.agroal.pool
   requestId : 
   realmId   : 
   message   : Datasource '<default>': Connection to benchmarks-postgresql-postgresql-ha-pgpool:5432 refused. Check that the hostname and port are correct and that the postmaster is accepting TCP/IP connections.

⚠️  [2026-07-03T04:44:08.911Z] WARN
   logger    : io.agroal.pool
   requestId : 
   realmId   : 
   message   : Datasource '<default>': Connection to benchmarks-postgresql-postgresql-ha-pgpool:5432 refused. Check that the hostname and port are correct and that the postmaster is accepting TCP/IP connections.

⚠️  [2026-07-03T04:44:07.883Z] WARN
   logger    : io.agroal.pool
   requestId : 
   realmId   : 
   message   : Datasource '<default>': Connection to benchmarks-postgresql-postgresql-ha-pgpool:5432 refused. 

In [9]:
# Cell 2.5: Check for blocked thread logs
# This appears when pgpool is down and Polaris tries to connect

print('Checking for Vert.x blocked thread warnings...')
wait_for_logs(15)  # Vert.x checks every 2s

hits = search_blocked_thread_errors(minutes_ago=2)
if hits:
    print('✅ Vert.x blocked thread detected:')
    print_logs(hits)
    print()
    print('Key insight:')
    print('  → Event loop thread blocked > 2000ms')
    print('  → Caused by: getConnection() waiting for pgpool')
    print('  → Impact: ALL requests on this thread are queued')
    print('  → Service becomes progressively unresponsive')
else:
    print('No blocked thread warning yet — pgpool may still be accessible')

Checking for Vert.x blocked thread warnings...
⏳ Waiting 15s for logs to reach OpenSearch...
✅ Vert.x blocked thread detected:
Found 2 log entries:

⚠️  [2026-07-03T04:43:55.073Z] WARN
   logger    : io.vertx.core.impl.BlockedThreadChecker
   requestId : 
   realmId   : 
   message   : Thread Thread[vert.x-eventloop-thread-1,5,main] has been blocked for 3563 ms, time limit is 2000 ms

⚠️  [2026-07-03T04:43:49.070Z] WARN
   logger    : io.vertx.core.impl.BlockedThreadChecker
   requestId : 
   realmId   : 
   message   : Thread Thread[vert.x-eventloop-thread-1,5,main] has been blocked for 3716 ms, time limit is 2000 ms


Key insight:
  → Event loop thread blocked > 2000ms
  → Caused by: getConnection() waiting for pgpool
  → Impact: ALL requests on this thread are queued
  → Service becomes progressively unresponsive


In [10]:
# Cell 3: Trigger errors during outage using pre-obtained token
import time

errors = []
request_ids = []
for i in range(3):
    try:
        r = requests.get(
            f'{BASE_MGMT}/catalogs',
            headers={
                'Authorization': f'Bearer {pre_outage_token}',
                'Polaris-Realm': REALM,
            },
            timeout=10    # ← increase timeout
        )
        errors.append(r.status_code)
        request_ids.append(get_request_id(r))
        # Safe print — don't assume JSON body
        try:
            msg = r.json().get('error', {}).get('message', '')[:80]
        except:
            msg = r.text[:80]
        print(f'Request {i+1}: {r.status_code} — {msg}')
    except Exception as e:
        errors.append(type(e).__name__)
        print(f'Request {i+1}: {type(e).__name__} — {str(e)[:80]}')
    time.sleep(2)   # ← increase delay

print(f'\nError codes during outage: {errors}')

Request 1: 401 — 
Request 2: 401 — 
Request 3: ConnectionError — HTTPConnectionPool(host='192.168.139.2', port=8181): Max retries exceeded with u

Error codes during outage: [401, 401, 'ConnectionError']


In [11]:
# Cell 4: Find connection error logs
wait_for_logs(5)
print('Searching for DB connection errors (last 2 min)...')
hits = search_db_errors(minutes_ago=2)
print_logs(hits)

# Also check for 500 errors
print('\nSearching for 500 errors...')
hits_500 = search_500_errors(minutes_ago=1)
print_logs(hits_500)

print(request_ids)
for request_id in request_ids:
    hits_request = search_and_debug(request_id, minutes_ago=1)
    print_logs(hits_request)

⏳ Waiting 5s for logs to reach OpenSearch...
Searching for DB connection errors (last 2 min)...
Found 10 log entries:

⚠️  [2026-07-03T04:45:39.018Z] WARN
   logger    : io.agroal.pool
   requestId : 
   realmId   : 
   message   : Datasource '<default>': Connection to benchmarks-postgresql-postgresql-ha-pgpool:5432 refused. Check that the hostname and port are correct and that the postmaster is accepting TCP/IP connections.

⚠️  [2026-07-03T04:45:37.995Z] WARN
   logger    : io.agroal.pool
   requestId : 
   realmId   : 
   message   : Datasource '<default>': Connection to benchmarks-postgresql-postgresql-ha-pgpool:5432 refused. Check that the hostname and port are correct and that the postmaster is accepting TCP/IP connections.

⚠️  [2026-07-03T04:45:36.972Z] WARN
   logger    : io.agroal.pool
   requestId : 
   realmId   : 
   message   : Datasource '<default>': Connection to benchmarks-postgresql-postgresql-ha-pgpool:5432 refused. Check that the hostname and port are correct and th

In [ ]:
# Cell 4.5: Assert the DB Connection Pool ALERT trigger actually fires
# Force-runs the monitor now (dryrun=True — evaluates both triggers'
# conditions without sending a real SMS/Messenger notification on every test
# run) and checks the ALERT trigger by name (trigger_results is keyed by
# trigger ID, not name — trigger_fired() scans by name instead). This is the
# signal that actually matters: not just that a log line with "connection
# refused" exists, but that the registered Monitor + Trigger from
# opensearch-monitor-alert-guide.md would actually have fired during this
# outage, using the tightened (2026-07-03) query.
if db_pool_monitor_id:
    result = alert_prov.execute_monitor_now(db_pool_monitor_id, dryrun=True)
    fired = alert_prov.trigger_fired(result, alert_prov.ALERT_TRIGGER_NAME)
    print(f'Monitor execution result: {result}')
    print(f'\nALERT trigger fired: {fired}')
    assert fired, 'DB Connection Pool ALERT trigger did NOT fire during a real pgpool outage — check the monitor query'
    print('✅ ALERT trigger confirmed firing on this outage (dryrun — no notification sent)')
    print('   Note: this proves detection only. Actual SMS/Messenger delivery to the')
    print('   internal Kafka webhook cannot be confirmed from this sandbox — verify manually')
    print('   once the real destination URL is wired in (see guide Step 1).')
else:
    print('⚠️  Skipped — monitor was not provisioned (see Cell 1.5)')

In [13]:
# Cell 5: Restore PostgreSQL (scale pgpool back up)
print('🔄 Restoring PostgreSQL...')

result = subprocess.run([
    'kubectl', 'scale', 'deployment',
    'benchmarks-postgresql-postgresql-ha-pgpool',
    '--replicas=1',
    '-n', 'datahub-hynix'
], capture_output=True, text=True)
print(result.stdout or result.stderr)

# Wait for pgpool ready
result = subprocess.run([
    'kubectl', 'rollout', 'status',
    'deployment/benchmarks-postgresql-postgresql-ha-pgpool',
    '-n', 'datahub-hynix',
    '--timeout=60s'
], capture_output=True, text=True)
print(result.stdout)
print('✅ PostgreSQL restored')

🔄 Restoring PostgreSQL...
deployment.apps/benchmarks-postgresql-postgresql-ha-pgpool scaled

Waiting for deployment "benchmarks-postgresql-postgresql-ha-pgpool" rollout to finish: 0 out of 1 new replicas have been updated...
Waiting for deployment "benchmarks-postgresql-postgresql-ha-pgpool" rollout to finish: 0 of 1 updated replicas are available...
deployment "benchmarks-postgresql-postgresql-ha-pgpool" successfully rolled out

✅ PostgreSQL restored


In [14]:
# Cell 6: Verify Polaris recovers automatically
import time
print('Testing Polaris auto-recovery...')

for attempt in range(5):
    time.sleep(3)
    try:
        r = requests.get(
            f'{BASE_MGMT}/catalogs',
            headers=h(),
            timeout=5
        )
        print(f'Attempt {attempt+1}: {r.status_code}')
        if r.status_code == 200:
            print('\n✅ Polaris recovered automatically')
            print('Note: This works because same DB data — no entity_version mismatch')
            print('See Notebook 16 for restore-from-backup scenario')
            break
    except Exception as e:
        print(f'Attempt {attempt+1}: {type(e).__name__}')

cleanup()
print('\n=== Summary ===')
print('Error:    500/503 during PostgreSQL outage')
print('Cause:    All JDBC connections broken')
print('Detect:   search_db_errors() → connection refused/broken pipe')
print('Fix:      PostgreSQL restore → Polaris auto-recovers (same data)')
print('Alert:    500 error spike → check PostgreSQL health first')
print('See NB16: If restored from backup → entity_version mismatch')

Testing Polaris auto-recovery...
Attempt 1: 200

✅ Polaris recovered automatically
Note: This works because same DB data — no entity_version mismatch
See Notebook 16 for restore-from-backup scenario
🧹 Cleaning up all test resources...
✅ Cleanup done

=== Summary ===
Error:    500/503 during PostgreSQL outage
Cause:    All JDBC connections broken
Detect:   search_db_errors() → connection refused/broken pipe
Fix:      PostgreSQL restore → Polaris auto-recovers (same data)
Alert:    500 error spike → check PostgreSQL health first
See NB16: If restored from backup → entity_version mismatch


In [ ]:
# Cell 7: Confirm the RESOLVED trigger fires after PostgreSQL recovers
# The ALERT trigger should now be false (pool is healthy again). The
# RESOLVED trigger is the new (2026-07-03) all-clear notification — it's
# designed to fire exactly once, on this first clean check right after the
# outage, since this is a PostgreSQL/infra-owned failure and whoever got
# paged needs an explicit "it's fixed" rather than just silence.
if db_pool_monitor_id:
    result = alert_prov.execute_monitor_now(db_pool_monitor_id, dryrun=True)
    alert_still_firing = alert_prov.trigger_fired(result, alert_prov.ALERT_TRIGGER_NAME)
    resolved_fired = alert_prov.trigger_fired(result, alert_prov.RESOLVED_TRIGGER_NAME)
    print(f'ALERT trigger still firing: {alert_still_firing}')
    print(f'RESOLVED trigger fired: {resolved_fired}')
    if not alert_still_firing:
        print('✅ ALERT correctly stopped firing after PostgreSQL recovery')
    else:
        print('⚠️  ALERT still firing — outage logs may still be within the 1m lookback window')
    if resolved_fired:
        print('✅ RESOLVED trigger fired — the all-clear notification would have been sent')
    else:
        print('⚠️  RESOLVED did not fire this run — either it already fired on a prior check,')
        print('   or the "previous" (now-2m..now-1m) window doesn\'t contain the outage anymore')
else:
    print('⚠️  Skipped — monitor was not provisioned (see Cell 1.5)')

In [ ]:
# Cell 7.5: Confirm blocked-thread ("concurrent") warnings are excluded
# Per 2026-07-03 direction: a Vert.x BlockedThreadChecker warning on its own
# is not a "database unavailable" signal — the client already sees the
# failed request and can retry, so it shouldn't page anyone. This asserts
# that decision is actually enforced by the monitor's query, not just
# documented.
import polaris_test_utils as ptu

fresh_blocked = search_blocked_thread_errors(minutes_ago=5)
print(f'Blocked-thread hits found this run: {len(fresh_blocked)}')

# Check every captured blocked-thread hit against the monitor's must-clauses
false_positives = [
    h for h in fresh_blocked
    if alert_prov.matches_db_pool_signal(
        h['_source'].get('loggerName', ''), h['_source'].get('message', '')
    )
]
assert not false_positives, f'Blocked-thread hits incorrectly match the DB pool query: {false_positives}'
print('✅ No blocked-thread hits match Monitor 7\'s query (correctly excluded)')

# Also check directly against the verbatim message captured earlier in this
# notebook run, so the assertion holds even if no blocked-thread event is in
# the current search window (timing-dependent).
known_blocked_thread_message = (
    'Thread Thread[vert.x-eventloop-thread-1,5,main] has been blocked for '
    '3563 ms, time limit is 2000 ms'
)
assert not alert_prov.matches_db_pool_signal(
    'io.vertx.core.impl.BlockedThreadChecker', known_blocked_thread_message
), 'Known blocked-thread message incorrectly matches the DB pool query'
print('✅ Known blocked-thread message (captured in Cell 6 this run) confirmed excluded')